In [1]:
import os
import pandas as pd
import numpy as np
from deep_translator import GoogleTranslator

# ============================================================
# KONFIGURASI
# ============================================================
SEED = 42
np.random.seed(SEED)

FILE_UTAMA   = '/mnt/extended-home/dzakaaufa/experiment/dataset/data_mix_inject_split.csv'
FILE_A       = '/mnt/extended-home/dzakaaufa/experiment/dataset/data_A_inject.csv'
FILE_B       = '/mnt/extended-home/dzakaaufa/experiment/dataset/filosofi_batik_v2.csv'
FILE_OUTPUT  = '/mnt/extended-home/dzakaaufa/experiment/dataset/final_dataset_with_philosophy1.csv'

# Koreksi typo nama kelas: key = nama di df_b (setelah normalize), value = nama di data utama
ALIAS_KELAS = {
    'truntum': 'tuntrum',   # typo di filosofi_batik_v2.csv
}

def normalize_kelas(s: str) -> str:
    """Lowercase + ganti spasi dan tanda hubung ke underscore."""
    return str(s).lower().strip().replace(' ', '_').replace('-', '_')

def basename_norm(path: str) -> str:
    """Ambil nama file dari path, lowercase, strip whitespace."""
    return os.path.basename(str(path)).strip().lower()

# ============================================================
# 1. MUAT SEMUA DATASET
# ============================================================
print("=" * 60)
print("Memuat dataset...")
df_master = pd.read_csv(FILE_UTAMA)
df_a      = pd.read_csv(FILE_A)
df_b      = pd.read_csv(FILE_B)

print(f"  data_mix_inject_split : {df_master.shape[0]} baris, {df_master['class'].nunique()} kelas")
print(f"  data_A_inject         : {df_a.shape[0]} baris")
print(f"  filosofi_batik_v2     : {df_b.shape[0]} baris (kelas B)")

# ============================================================
# 2. PROSES DATASET A — Translasi HISTORY & Mapping via Basename
# ============================================================
print("\n[DATASET A] Menerjemahkan kolom HISTORY (id → en)...")

translator = GoogleTranslator(source='id', target='en')

def translate_safe(text: str) -> str | None:
    if pd.isna(text) or str(text).strip() == "":
        return None
    try:
        return translator.translate(str(text))
    except Exception as e:
        print(f"  [WARN] Gagal menerjemahkan: {str(text)[:60]}... | Error: {e}")
        return None

df_a['philosophy_a'] = df_a['HISTORY'].apply(translate_safe)

# Buat kolom basename untuk join yang aman di semua environment
df_a['basename_key']      = df_a['Image_Path'].apply(basename_norm)
df_master['basename_key'] = df_master['image_path'].apply(basename_norm)

# Buat lookup dict: basename → philosophy_a
lookup_a = df_a.set_index('basename_key')['philosophy_a'].to_dict()

# Mapping ke df_master — hanya baris yang memiliki entri di lookup_a
df_master['philosophy_a'] = df_master['basename_key'].map(lookup_a)

matched_a = df_master['philosophy_a'].notna().sum()
print(f"  Berhasil di-inject philosophy_a: {matched_a}/{len(df_a)} baris")

# ============================================================
# 3. PROSES DATASET B — Distribusi 3 Variasi & Philosophy Core
# ============================================================
print("\n[DATASET B] Mendistribusikan variasi filosofi dan menginjeksi Philosophy_Core...")

# Normalisasi kelas di kedua sisi, lalu terapkan alias typo
df_b['class_norm'] = df_b['Class'].apply(normalize_kelas).replace(ALIAS_KELAS)
df_master['class_norm'] = df_master['class'].apply(normalize_kelas)

# Inisialisasi placeholder kolom baru
df_master['philosophy_b'] = None  
df_master['philosophy_core'] = None  # [BARU] Kosong secara default (termasuk untuk Dataset A)

kelas_b_list = df_b['class_norm'].tolist()
report_b = []

for cls_norm in kelas_b_list:
    baris_variasi = df_b[df_b['class_norm'] == cls_norm]
    if baris_variasi.empty:
        print(f"  [WARN] Kelas '{cls_norm}' tidak ditemukan di df_b — dilewati.")
        continue

    row_b = baris_variasi.iloc[0]
    v1, v2, v3 = row_b['Variasi 1'], row_b['Variasi 2'], row_b['Variasi 3']
    
    # [BARU] Ambil nilai keyword utama dari kolom Philosophy_Core
    core_keyword = row_b.get('Philosophy_Core', None)

    # Ambil semua baris di data utama untuk kelas ini
    idx_kelas = df_master[df_master['class_norm'] == cls_norm].index.tolist()
    jumlah_data = len(idx_kelas)

    if jumlah_data == 0:
        print(f"  [WARN] Kelas '{cls_norm}' tidak ditemukan di data utama — dilewati.")
        report_b.append({'kelas': cls_norm, 'n_baris': 0, 'status': 'TIDAK DITEMUKAN'})
        continue

    # Distribusi merata: jumlah_per_variasi tiap variasi, sisa masuk ke v3
    n = jumlah_data // 3
    sisa = jumlah_data % 3
    distribusi = [v1] * n + [v2] * n + [v3] * (n + sisa)

    # Seed sudah di-set global → shuffle reprodusibel
    np.random.shuffle(distribusi)

    # Inject data ke master dataframe
    df_master.loc[idx_kelas, 'philosophy_b'] = distribusi
    df_master.loc[idx_kelas, 'philosophy_core'] = core_keyword  # [BARU] Mapping berdasarkan kelas

    report_b.append({
        'kelas'  : cls_norm,
        'n_baris': jumlah_data,
        'v1'     : n,
        'v2'     : n,
        'v3'     : n + sisa,
        'status' : 'OK'
    })

# Laporan distribusi Dataset B
df_report = pd.DataFrame(report_b)
print("\n  Laporan distribusi Dataset B:")
print(df_report.to_string(index=False))

# ============================================================
# 4. PENGGABUNGAN AKHIR
# ============================================================
print("\n[FINAL] Menggabungkan philosophy_a dan philosophy_b...")

# Prioritas: philosophy_a (per-gambar, Dataset A) > philosophy_b (per-kelas, Dataset B)
df_master['philosophy_en'] = df_master['philosophy_a'].fillna(df_master['philosophy_b'])

# ============================================================
# 5. VALIDASI COVERAGE
# ============================================================
total       = len(df_master)
filled      = df_master['philosophy_en'].notna().sum()
nan_rows    = total - filled
nan_classes = df_master[df_master['philosophy_en'].isna()]['class'].value_counts()

print(f"\n  Total baris        : {total}")
print(f"  Terisi philosophy  : {filled} ({filled/total*100:.1f}%)")
print(f"  NaN philosophy_en  : {nan_rows} ({nan_rows/total*100:.1f}%)")
if nan_rows > 0:
    print(f"\n  [WARN] Kelas dengan NaN:\n{nan_classes.to_string()}")

# ============================================================
# 6. BERSIHKAN & SIMPAN
# ============================================================
# Pastikan kolom 'philosophy_core' tidak ikut terhapus di sini
kolom_temp = ['philosophy_a', 'philosophy_b', 'class_norm', 'basename_key']
df_master.drop(columns=[c for c in kolom_temp if c in df_master.columns], inplace=True)

# [BARU] Menyertakan 'philosophy_core' ke dalam urutan kolom target akhir
urutan_kolom = ['name', 'class', 'image_path', 'caption_en', 'split', 'philosophy_en', 'philosophy_core']
if all(c in df_master.columns for c in urutan_kolom):
    df_master = df_master[urutan_kolom]

os.makedirs(os.path.dirname(FILE_OUTPUT), exist_ok=True)
df_master.to_csv(FILE_OUTPUT, index=False)

print(f"\n{'='*60}")
print(f"[SELESAI] File disimpan ke: {FILE_OUTPUT}")
print(f"          Shape akhir    : {df_master.shape}")

Memuat dataset...
  data_mix_inject_split : 3327 baris, 24 kelas
  data_A_inject         : 327 baris
  filosofi_batik_v2     : 20 baris (kelas B)

[DATASET A] Menerjemahkan kolom HISTORY (id → en)...
  Berhasil di-inject philosophy_a: 327/327 baris

[DATASET B] Mendistribusikan variasi filosofi dan menginjeksi Philosophy_Core...

  Laporan distribusi Dataset B:
        kelas  n_baris  v1  v2  v3 status
       betawi      150  50  50  50     OK
bokor_kencono      150  50  50  50     OK
      buketan      150  50  50  50     OK
        dayak      150  50  50  50     OK
    jlamprang      150  50  50  50     OK
       kawung      150  50  50  50     OK
        liong      150  50  50  50     OK
 mega_mendung      150  50  50  50     OK
       parang      150  50  50  50     OK
   sekarjagad      150  50  50  50     OK
    sidoluhur      150  50  50  50     OK
    sidomukti      150  50  50  50     OK
    sidomulyo      150  50  50  50     OK
 singa_barong      150  50  50  50     OK
     s

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

def create_uniform_stratified_split(csv_path, output_path):
    # Load dataset
    df = pd.read_csv(csv_path)
    
    # Normalisasi nama kolom agar seragam (Antisipasi kapitalisasi)
    df = df.rename(columns={
        "Nama": "name", "NAME": "name",
        "CLASS": "class", "Class": "class", "CLASS ": "class",
        "Image Path": "image_path", "IMAGE_PATH": "image_path",
        "CAPTION_EN": "caption_en", "Caption_En": "caption_en"
    })
    
    # [BARU] Drop kolom 'split' jika sudah ada di file asli
    if 'split' in df.columns:
        df = df.drop(columns=['split'])
        print("Sistem mendeteksi kolom 'split' lama. Kolom tersebut telah dihapus untuk dibuat ulang.")
    
    # Pastikan data class tidak ada yang kosong/NaN agar tidak error saat stratify
    df = df.dropna(subset=['class'])
    
    # =========================================================
    # PROSES SPLIT DATASET (Rasio 80:10:10 - Stratified)
    # =========================================================
    
    # 1. Ambil 20% data untuk gabungan Val & Test (Sisa 80% otomatis jadi Train)
    train_df, temp_df = train_test_split(
        df, test_size=0.20, stratify=df['class'], random_state=42
    )
    
    # 2. Bagi 20% data tadi menjadi 50:50 untuk Validation (10%) dan Test (10%)
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['class'], random_state=42
    )
    
    # =========================================================
    # PEMBERIAN LABEL DAN PENGGABUNGAN
    # =========================================================
    train_df['split'] = 'train'
    val_df['split'] = 'val'
    test_df['split'] = 'test'
    
    # Gabungkan semua menjadi satu berkas master
    master_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    master_df.to_csv(output_path, index=False)
    
    print("\n" + "="*50)
    print(" PIPELINE STRATIFIED SPLIT (80:10:10) BERHASIL")
    print("="*50)
    print(f"Total DATA TRAIN (80%) : {len(train_df)}")
    print(f"Total DATA VAL   (10%) : {len(val_df)}")
    print(f"Total DATA TEST  (10%) : {len(test_df)}")
    print(f"Grand Total Dataset    : {len(master_df)}")
    print("="*50)

# Jalankan fungsi
create_uniform_stratified_split(
    "/mnt/extended-home/dzakaaufa/experiment/dataset/final_dataset_with_philosophy1.csv", 
    "/mnt/extended-home/dzakaaufa/experiment/dataset/final_dataset_split1.csv"
)

Sistem mendeteksi kolom 'split' lama. Kolom tersebut telah dihapus untuk dibuat ulang.

 PIPELINE STRATIFIED SPLIT (80:10:10) BERHASIL
Total DATA TRAIN (80%) : 2661
Total DATA VAL   (10%) : 333
Total DATA TEST  (10%) : 333
Grand Total Dataset    : 3327
